In [ ]:
# NLLB ON PHOENIX !!!!!!!!!!!!!!

In [ ]:
import pandas as pd         

file_path = "PHOENIX_TRAIN.csv" # DataFrame structure: encodes a spreadsheet in Python. Two-dimensional data structure: rows and columns. Each column has a header and each row has an index
df = pd.read_csv(file_path)     # Creates a DataFrame variable 'df', using the read_csv command to load the "PHOENIX_TRAIN.csv" file from the root directory
df.head(15)

In [ ]:
import sys

print("--- STARTING ENVIRONMENT CHECK ---")
print(f"Using Python version: {sys.version}\n")

# --- Checking Main Libraries ---

try:
    import transformers
    print(f"Transformers OK. Installed version: {transformers.__version__}")
except ImportError:
    print("ERROR: 'transformers' library not found.")

try:
    import torch
    print(f"PyTorch (torch) OK. Installed version: {torch.__version__}")
except ImportError:
    print("ERROR: 'torch' library not found.")

try:
    import datasets
    print(f"Datasets OK. Installed version: {datasets.__version__}")
except ImportError:
    print("ERROR: 'datasets' library not found.")

try:
    import accelerate
    print(f"Accelerate OK. Installed version: {accelerate.__version__}")
except ImportError:
    print("ERROR: 'accelerate' library not found.")

try:
    import pandas
    print(f"Pandas OK. Installed version: {pandas.__version__}")
except ImportError:
    print("ERROR: 'pandas' library not found.")

try:
    import sacrebleu
    print(f"Sacrebleu OK. Installed version: {sacrebleu.__version__}")
except ImportError:
    print("ERROR: 'sacrebleu' library not found.")

try:
    import evaluate
    print(f"Evaluate OK. Installed version: {evaluate.__version__}")
except ImportError:
    print("ERROR: 'evaluate' library not found.")


# --- CRITICAL CHECK: GPU Access ---
print("\n--- CHECKING GPU (CUDA) ACCESS ---")
try:
    import torch
    if torch.cuda.is_available():
        print(f"SUCCESS! PyTorch found the GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("WARNING: PyTorch is installed, but could NOT find your local GPU.")
except Exception as e:
    print(f"ERROR: Failed to check GPU status with PyTorch: {e}")

print("\n--- VERIFICATION COMPLETED ---")

In [ ]:
import os

# The script to be executed
generator_script = "gen_variants.py"

# Step 1: Check if the input files exist
# The "!" executes a shell command directly from the Jupyter Notebook and returns the output
print("Checking input files:")
!ls -l PHOENIX_*.csv # ls -> list; -l: long format; PHOENIX_*.csv matches any file starting with PHOENIX_ and ending with .csv

# --- Step 2: Execute the script ---
print("\nExecuting the gen_variants.py script...")
!python {generator_script}   # Calls the Python interpreter in the shell, passing the {generator_script} variable. The shell reads "python gen_variants.py"
print("\nScript execution completed!")

# --- Step 3: Verify the output ---
print("\nContents of the 'variants' folder (Should display 6 CSV files and 2 JSON/TXT files):")
!ls -l variants/ # Lists the files inside the variants folder

In [1]:
# ==============================================================================
# PHASE 1 - TRAINING WITH DATA AUGMENTATION - NLLB (600M) - PHOENIX-14T
# ==============================================================================

import os                                   
import json
import random
import numpy as np                          
import torch                                
import pandas as pd                         
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from bitsandbytes.optim import AdamW8bit    

from transformers import (                  
    AutoTokenizer, 
    get_scheduler, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    AutoConfig
)
import datasets                             # Provides dataset handling structures
datasets.config.TORCHVISION_AVAILABLE = False
import sys
if "torchvision" in sys.modules:
    del sys.modules["torchvision"]                       
import evaluate                       
from accelerate import Accelerator    

# Creates text labels to avoid typing paths
DATASET_PATH = "./variants"
dataset_variant = "lower"
MSRC_LANG = "de"
AUX_LANG = "dgs"
RUN = "Vanilla"                      # Updated to reflect the model

# Hyperparameters
BATCH_SIZE = 4            
gradient_accumulation_steps = 4 
N_EPOCHS = 200                                                                   
LEARNING_RATE = 5e-5                                                                                    
LR_SCHEDULER = "linear"    
SRC_LANG = 'de'   
TGT_LANG = 'dgs'
MAX_INPUT_LENGTH = 256     
MAX_TGT_LENGTH = 256       

VARIANT = '.' + dataset_variant if dataset_variant != 'plain' else '' 

### DATASETS 
TRAIN_PATH = os.path.join(DATASET_PATH, f"de-dgs.augmented.train.csv") 
TEST_PATH = os.path.join(DATASET_PATH, f"de-dgs{VARIANT}.test.csv") 
DEV_PATH = os.path.join(DATASET_PATH, f"de-dgs{VARIANT}.val.csv") 
VAR_NAME = dataset_variant
output_dir = f'runs/{MSRC_LANG}-{AUX_LANG}-{VAR_NAME}-run{RUN}' 
os.makedirs(output_dir, exist_ok=True) 

# Hardware Configuration
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==============================================================================
# 1. POST-PROCESSING FUNCTION: "TRANSLATING THE TRANSLATION" (IDS -> SENTENCES)
# ==============================================================================
def postprocess(predictions, labels):
    predictions = predictions.cpu().numpy() 
    labels = labels.cpu().numpy()           
    decoded_preds = tokenizers['pt_X'].batch_decode(predictions, skip_special_tokens=True) 
    labels = np.where(labels != -100, labels, tokenizers['pt_X'].pad_token_id) 
    decoded_labels = tokenizers['pt_X'].batch_decode(labels, skip_special_tokens=True) 
    decoded_preds = [pred.strip() for pred in decoded_preds] 
    decoded_labels = [[label.strip()] for label in decoded_labels] 
    return decoded_preds, decoded_labels 

# ==============================================================================
# 2. DATA PREPARATION: THE NEW NLLB ENGINE
# ==============================================================================
vocab_file = os.path.join(DATASET_PATH, 'gloss_vocab.txt') 
with open(vocab_file, 'r') as f:  
    gloss_vocab = [line.strip() for line in f if line.strip()] 

# CHANGE 1: Model alteration and Tokenizer Instantiation with src_lang
MODEL_NAME = "facebook/nllb-200-distilled-600M" 
print(f" Loading Tokenizer: {MODEL_NAME}")

# CHANGE 2: NLLB requires setting src_lang directly during instantiation using its phonetic code.
# For German, the exact NLLB code is "deu_Latn".
tokenizers = {'pt_X': AutoTokenizer.from_pretrained(MODEL_NAME, src_lang="deu_Latn")} 

# Adds the gloss list to the NLLB vocabulary.
tokenizers['pt_X'].add_tokens(list(gloss_vocab)) 

print(" Loading CSVs...") 
raw_datasets = datasets.DatasetDict({  
    'train': datasets.Dataset.from_pandas(pd.read_csv(TRAIN_PATH)),
    'validation': datasets.Dataset.from_pandas(pd.read_csv(DEV_PATH)),
    'test': datasets.Dataset.from_pandas(pd.read_csv(TEST_PATH))
})

raw_datasets = raw_datasets.filter(lambda ex: pd.notna(ex['de']) and pd.notna(ex['dgs']) and len(str(ex['de']).strip()) > 0) 

def preprocess_function(batch):
    inputs = [str(text) for text in batch["de"]] 
    targets = [str(text) for text in batch["dgs"]] 
    
    # 1. NLLB Style Tokenization for inputs (German)
    model_inputs = tokenizers['pt_X'](inputs, max_length=MAX_INPUT_LENGTH, truncation=True) 
    
    # CRITICAL CHANGE: NLLB requires using the context manager to tokenize the targets.
    # Without this, it does not know how to wrap the glosses and returns a NoneType error.
    # Moreover, we ensure it understands that the target also belongs to the same latent space.
    tokenizers['pt_X'].tgt_lang = "deu_Latn" # We force the output to map to the German phonetics
    
    with tokenizers['pt_X'].as_target_tokenizer():
        labels = tokenizers['pt_X'](targets, max_length=MAX_TGT_LENGTH, truncation=True) 
        
    model_inputs["labels"] = labels["input_ids"] 
    return model_inputs

print(" Configuring Tokenization...")
tokenized_datasets = raw_datasets.map(   
    preprocess_function, 
    batched=True,  
    remove_columns=raw_datasets["train"].column_names 
)
tokenized_datasets.set_format("torch") 

# ==============================================================================
# 3. TRAINING PREPARATION: CHECKPOINTS & ARCHITECTURE INITIALIZATION
# ==============================================================================
best_bleu = 0.0
checkpoint_config = os.path.join(output_dir, "config.json") 

if os.path.exists(checkpoint_config):   
    print(f"\n CHECKPOINT FOUND AT: {output_dir}")
    model = AutoModelForSeq2SeqLM.from_pretrained(output_dir) 
    try:
        with open(os.path.join(output_dir, 'stats.json'), 'r') as f: 
            old_stats = json.load(f)                                 
            if 'bleu' in old_stats and len(old_stats['bleu']) > 0:   
                best_bleu = max(old_stats['bleu'])                   
            print(f"   -> History recovered. Previous best BLEU: {best_bleu:.2f}")
    except:
        pass 
else: 
    print(f" STARTING TRAINING FROM SCRATCH: {MODEL_NAME}")
    config = AutoConfig.from_pretrained(MODEL_NAME)   
    
    # CHANGE 3: The Dropout configuration in NLLB is compatible, but it is important to ensure
    # that the activation_dropout is not overwritten if not explicit.
    config.dropout = 0                                                                           
    config.attention_dropout = 0                                                                 
    
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, config=config) 
    
    # THEORETICAL CRITIQUE: NLLB has a colossal vocabulary (~256k native tokens).
    # The resizing below will make the Embeddings matrix even larger than mBART's.
    # This makes your DLR (Differential Learning Rates) strategy even more crucial and defensible in the paper.
    model.resize_token_embeddings(len(tokenizers['pt_X'])) 

# ==============================================================================
# DIFFERENTIAL OPTIMIZER (Differential LRs)
# ==============================================================================
embedding_params = model.get_input_embeddings().parameters() 
embedding_param_ids = {id(p) for p in embedding_params}      

body_params = [p for p in model.parameters() if id(p) not in embedding_param_ids] 

optimizer_grouped_parameters = [ 
    {"params": body_params, "lr": LEARNING_RATE},                                  
    {"params": model.get_input_embeddings().parameters(), "lr": LEARNING_RATE * 1}, 
]

optimizer = AdamW8bit(optimizer_grouped_parameters, weight_decay=0.05) 

data_collator = DataCollatorForSeq2Seq(tokenizers['pt_X'], model=model) 

train_dataloader = DataLoader(tokenized_datasets["train"], shuffle=True, collate_fn=data_collator, batch_size=BATCH_SIZE)
eval_dataloader = DataLoader(tokenized_datasets["validation"], collate_fn=data_collator, batch_size=BATCH_SIZE)

accelerator = Accelerator(gradient_accumulation_steps=gradient_accumulation_steps)
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

num_training_steps = (len(train_dataloader) // gradient_accumulation_steps) * N_EPOCHS
warmup_steps = int(num_training_steps * 0)              

lr_scheduler = get_scheduler(
    LR_SCHEDULER, 
    optimizer=optimizer, 
    num_warmup_steps=warmup_steps, 
    num_training_steps=num_training_steps
)
metric = evaluate.load("sacrebleu")

# CHANGE 4: Safe extraction of the Forced Language ID (BOS).
# NLLB requires the decoder to start with the exact target language ID ("deu_Latn").
forced_bos_token_id = tokenizers['pt_X'].convert_tokens_to_ids("deu_Latn")

# ==============================================================================
# 4. TRAINING LOOP (KEPT INTACT FROM HERE)
# ==============================================================================
patience = 10                                                                                      
patience_counter = 0

print(f"\n STARTING TRAINING | Epochs: {N_EPOCHS} | Best BLEU to beat: {best_bleu:.2f}")
progress_bar = tqdm(range(num_training_steps))
writer = SummaryWriter(log_dir=f'{output_dir}/logs')

stats = {'bleu': []}

for epoch in range(N_EPOCHS):
    # --- TRAINING ---
    model.train()
    total_loss = 0
    
    loss_fct = torch.nn.CrossEntropyLoss(label_smoothing=0, ignore_index=-100)                 
    
    for batch in train_dataloader:
        outputs = model(**batch)
        
        logits = outputs.logits
        labels = batch["labels"]
        
        # OPTIMIZED MEMORY LOSS CALCULATION 
        loss = loss_fct(logits.transpose(1, 2), labels)
        
        loss = loss / gradient_accumulation_steps
        accelerator.backward(loss)
        
        if (progress_bar.n + 1) % gradient_accumulation_steps == 0:
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            
        total_loss += loss.item() * gradient_accumulation_steps
        progress_bar.update(1)
    
    avg_loss = total_loss / len(train_dataloader)
    writer.add_scalar('Training/Loss', avg_loss, epoch)

    # --- VALIDATION ---
    model.eval()
    print(f"\n--- Epoch {epoch+1} ---")   
    for batch in tqdm(eval_dataloader, desc="Validation", leave=False):
        with torch.no_grad():
            generated_tokens = accelerator.unwrap_model(model).generate(
                batch["input_ids"], 
                attention_mask=batch["attention_mask"], 
                max_length=MAX_TGT_LENGTH,
                forced_bos_token_id=forced_bos_token_id,
                num_beams=1                                                                      
            )
        
        labels = batch["labels"]
        generated_tokens = accelerator.pad_across_processes(generated_tokens, dim=1, pad_index=tokenizers['pt_X'].pad_token_id)
        labels = accelerator.pad_across_processes(labels, dim=1, pad_index=-100)
        
        predictions_gathered = accelerator.gather(generated_tokens)
        labels_gathered = accelerator.gather(labels)
        decoded_preds, decoded_labels = postprocess(predictions_gathered, labels_gathered)
        metric.add_batch(predictions=decoded_preds, references=decoded_labels)

    # PURIST EVALUATION
    results = metric.compute(tokenize='none', lowercase=True)
    current_bleu = results['score']
    
    print(f" Loss: {avg_loss:.4f} | Validation BLEU: {current_bleu:.2f} (Best: {best_bleu:.2f})")
    writer.add_scalar('BLEU/Validation', current_bleu, epoch)
    stats['bleu'].append(current_bleu)

    if current_bleu > best_bleu:
        print(f" New Record! (Old: {best_bleu:.2f} -> New: {current_bleu:.2f}) Saving model...")
        best_bleu = current_bleu
        patience_counter = 0
        accelerator.wait_for_everyone()
        accelerator.unwrap_model(model).save_pretrained(output_dir, save_function=accelerator.save)
        tokenizers['pt_X'].save_pretrained(output_dir)
        
        with open(os.path.join(output_dir, 'stats.json'), 'w') as f:
            json.dump(stats, f)
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{patience}")
        
    if patience_counter >= patience:
        print(" Early stopping. Model is no longer learning.")
        break

writer.close()
print(" Training completed.")

 Loading Tokenizer: facebook/nllb-200-distilled-600M
 Loading CSVs...


Filter:   0%|          | 0/14192 [00:00<?, ? examples/s]

Filter:   0%|          | 0/519 [00:00<?, ? examples/s]

Filter:   0%|          | 0/642 [00:00<?, ? examples/s]

 Configuring Tokenization...


Map:   0%|          | 0/14192 [00:00<?, ? examples/s]

... (50 outputs left out)

 Loss: 0.1508 | Validation BLEU: 14.40 (Best: 13.93)
 New Record! (Old: 13.93 -> New: 14.40) Saving model...

--- Epoch 21 ---


Validation:   0%|          | 0/130 [00:00<?, ?it/s]

 Loss: 0.1383 | Validation BLEU: 13.52 (Best: 14.40)
No improvement. Patience: 1/10


KeyboardInterrupt: 

In [ ]:
##################################################################################

In [ ]:
##################################################################################

In [2]:
# ==============================================================================
# TRAINING SCRIPT - PHASE 2 (EXPERT FINE-TUNING) NLLB (600M) PHOENIX-14T
# ==============================================================================

import os
import json
import random
import numpy as np
import torch
import pandas as pd
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from bitsandbytes.optim import AdamW8bit

from transformers import (
    AutoTokenizer, 
    get_scheduler, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    AutoConfig
)
import datasets
from accelerate import Accelerator
import evaluate

# --- 1. PATH CONFIGURATION ---
DATASET_PATH = "./variants"
dataset_variant = "lower"
MSRC_LANG = "de"
AUX_LANG = "dgs"

# CHANGE: Point to the exact folder generated by Phase 1 of NLLB
PREV_RUN_DIR = f'runs/{MSRC_LANG}-{AUX_LANG}-{dataset_variant}-runVanilla' 
RUN = "2FASE_ESPECIALISTA_NLLB"                                                     

BATCH_SIZE = 4                  
gradient_accumulation_steps = 4 
N_EPOCHS = 40                     
LEARNING_RATE = 1e-5              
LR_SCHEDULER = "linear"
WEIGHT_DECAY = 0.1
SRC_LANG = 'de'   
TGT_LANG = 'dgs'
MAX_INPUT_LENGTH = 256
MAX_TGT_LENGTH = 256

VARIANT = '.' + dataset_variant if dataset_variant != 'plain' else '' 
TRAIN_PATH = os.path.join(DATASET_PATH, f"de-dgs{VARIANT}.train.csv") 
TEST_PATH = os.path.join(DATASET_PATH, f"de-dgs{VARIANT}.test.csv")
DEV_PATH = os.path.join(DATASET_PATH, f"de-dgs{VARIANT}.val.csv") 

output_dir = f'runs/{MSRC_LANG}-{AUX_LANG}-{dataset_variant}-run{RUN}'
os.makedirs(output_dir, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==============================================================================
# 2. AUXILIARY FUNCTIONS AND DATA PREPARATION
# ==============================================================================
print(f" Loading Tokenizer from Phase 1 pre-trained model: {PREV_RUN_DIR}")

# CHANGE: NLLB needs to know src_lang immediately upon instantiation to avoid corrupting the input embedding.
tokenizers = {'pt_X': AutoTokenizer.from_pretrained(PREV_RUN_DIR, src_lang="deu_Latn")}

# CHANGE: Extraction of NLLB-specific forced BOS (Beginning of Sentence).
forced_bos_token_id = tokenizers['pt_X'].convert_tokens_to_ids("deu_Latn")

print(" Loading ORIGINAL CSVs (No Augmentation)...")
raw_datasets = datasets.DatasetDict({
    'train': datasets.Dataset.from_pandas(pd.read_csv(TRAIN_PATH)),
    'validation': datasets.Dataset.from_pandas(pd.read_csv(DEV_PATH)),
    'test': datasets.Dataset.from_pandas(pd.read_csv(TEST_PATH))
})

raw_datasets = raw_datasets.filter(lambda ex: pd.notna(ex['de']) and pd.notna(ex['dgs']) and len(str(ex['de']).strip()) > 0)
raw_datasets = raw_datasets.map(
    lambda ex: {"translation": {SRC_LANG: str(ex[SRC_LANG]), TGT_LANG: str(ex[TGT_LANG])}},
    remove_columns=[SRC_LANG, TGT_LANG]
)

def preprocess_function(examples):
    inputs = [str(ex[SRC_LANG]) for ex in examples["translation"]]
    targets = [str(ex[TGT_LANG]) for ex in examples["translation"]]
    
    # The tokenization logic works exactly like mBART for NLLB if instantiated correctly
    model_inputs = tokenizers['pt_X'](inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizers['pt_X'](text_target=targets, max_length=MAX_TGT_LENGTH, truncation=True)
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print(" Tokenizing...")
tokenized_datasets = raw_datasets.map(
    preprocess_function, 
    batched=True,
    remove_columns=raw_datasets["train"].column_names
)
tokenized_datasets.set_format("torch")

# ==============================================================================
# 3. MODEL & OPTIMIZER
# ==============================================================================
print(f"\n Loading Model: {PREV_RUN_DIR}")
config = AutoConfig.from_pretrained(PREV_RUN_DIR)
config.dropout = 0.2  # Reduced for Expert Phase
model = AutoModelForSeq2SeqLM.from_pretrained(PREV_RUN_DIR, config=config)

# 1. Isolates Embeddings
embedding_params = model.get_input_embeddings().parameters()
embedding_param_ids = {id(p) for p in embedding_params}
    
# 2. Isolates NLLB Body
body_params = [p for p in model.parameters() if id(p) not in embedding_param_ids]
    
# 3. Differential Rates: Cools the body, keeps embeddings slightly higher
optimizer_grouped_parameters = [
    {"params": body_params, "lr": LEARNING_RATE},                            
    {"params": model.get_input_embeddings().parameters(), "lr": LEARNING_RATE * 10}, 
]

# Injecting more aggressive Weight Decay
optimizer = AdamW8bit(optimizer_grouped_parameters, weight_decay=WEIGHT_DECAY)

data_collator = DataCollatorForSeq2Seq(tokenizers['pt_X'], model=model)

train_dataloader = DataLoader(tokenized_datasets["train"], shuffle=True, collate_fn=data_collator, batch_size=BATCH_SIZE)
eval_dataloader = DataLoader(tokenized_datasets["validation"], collate_fn=data_collator, batch_size=BATCH_SIZE)

accelerator = Accelerator(gradient_accumulation_steps=gradient_accumulation_steps)
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

# Exact calculation of real optimization steps
actual_training_steps = (len(train_dataloader) // gradient_accumulation_steps) * N_EPOCHS

# 10% WARM-UP: The model will spend 10% of training just "waking up" the weights
warmup_steps = int(actual_training_steps * 0.10)

lr_scheduler = get_scheduler(
    LR_SCHEDULER, 
    optimizer=optimizer, 
    num_warmup_steps=warmup_steps, 
    num_training_steps=actual_training_steps
)

metric = evaluate.load("sacrebleu")

def postprocess(predictions, labels):
    predictions = predictions.cpu().numpy()
    labels = labels.cpu().numpy()
    decoded_preds = tokenizers['pt_X'].batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizers['pt_X'].pad_token_id)
    decoded_labels = tokenizers['pt_X'].batch_decode(labels, skip_special_tokens=True)
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    return decoded_preds, decoded_labels

# ==============================================================================
# 4. TRAINING LOOP (PHASE 2)
# ==============================================================================
best_bleu = 0.0
patience = 10                                                                                                                                                                                                            
patience_counter = 0

print(f"\n START PHASE 2 | Epochs: {N_EPOCHS} | LR: {LEARNING_RATE}")
progress_bar = tqdm(range(actual_training_steps))
writer = SummaryWriter(log_dir=f'{output_dir}/logs')

stats = {'bleu': []}

for epoch in range(N_EPOCHS):
    model.train()
    total_loss = 0
    
    # Label Smoothing enabled to prevent Overfitting in Phase 2
    loss_fct = torch.nn.CrossEntropyLoss(label_smoothing=0.1, ignore_index=-100)
    
    for batch in train_dataloader:
        outputs = model(**batch)
        logits = outputs.logits
        labels = batch["labels"]
        
        # Safe manual memory calculation
        loss = loss_fct(logits.transpose(1, 2), labels)
        
        loss = loss / gradient_accumulation_steps
        accelerator.backward(loss)
        
        if (progress_bar.n + 1) % gradient_accumulation_steps == 0:
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            
        total_loss += loss.item() * gradient_accumulation_steps
        progress_bar.update(1)
    
    avg_loss = total_loss / len(train_dataloader)
    writer.add_scalar('Training/Loss', avg_loss, epoch)

    model.eval()
    print(f"\n--- Epoch {epoch+1} ---")
    
    for batch in tqdm(eval_dataloader, desc="Validation", leave=False):
        with torch.no_grad():
            generated_tokens = accelerator.unwrap_model(model).generate(
                batch["input_ids"], 
                attention_mask=batch["attention_mask"], 
                max_length=MAX_TGT_LENGTH,
                forced_bos_token_id=forced_bos_token_id,
                num_beams=5
            )
        
        labels = batch["labels"]
        generated_tokens = accelerator.pad_across_processes(generated_tokens, dim=1, pad_index=tokenizers['pt_X'].pad_token_id)
        labels = accelerator.pad_across_processes(labels, dim=1, pad_index=-100)
        
        predictions_gathered = accelerator.gather(generated_tokens)
        labels_gathered = accelerator.gather(labels)
        decoded_preds, decoded_labels = postprocess(predictions_gathered, labels_gathered)
        metric.add_batch(predictions=decoded_preds, references=decoded_labels)

    # Strict evaluation identical to Phase 1
    results = metric.compute(tokenize='none', lowercase=True)
    current_bleu = results['score']
    
    print(f" Loss: {avg_loss:.4f} | Validation BLEU: {current_bleu:.2f} (Best: {best_bleu:.2f})")
    writer.add_scalar('BLEU/Validation', current_bleu, epoch)
    stats['bleu'].append(current_bleu)

    if current_bleu > best_bleu:
        print(f" New Record! Saving model ...")
        best_bleu = current_bleu
        patience_counter = 0
        accelerator.wait_for_everyone()
        accelerator.unwrap_model(model).save_pretrained(output_dir, save_function=accelerator.save)
        tokenizers['pt_X'].save_pretrained(output_dir)
        
        with open(os.path.join(output_dir, 'stats.json'), 'w') as f:
            json.dump(stats, f)
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{patience}")
        
    if patience_counter >= patience:
        print(" Early stopping. Model reached its limit.")
        break

writer.close()
print(" Phase 2 training completed. Proceed to the inference cell.")

 Loading Tokenizer from Phase 1 pre-trained model: runs/de-dgs-lower-runVanilla
 Loading ORIGINAL CSVs (No Augmentation)...


Filter:   0%|          | 0/7096 [00:00<?, ? examples/s]

Filter:   0%|          | 0/519 [00:00<?, ? examples/s]

Filter:   0%|          | 0/642 [00:00<?, ? examples/s]

Map:   0%|          | 0/7096 [00:00<?, ? examples/s]

Map:   0%|          | 0/519 [00:00<?, ? examples/s]

... (4 outputs left out)

Map:   0%|          | 0/642 [00:00<?, ? examples/s]


 Loading Model: runs/de-dgs-lower-runVanilla

 START PHASE 2 | Epochs: 40 | LR: 1e-05


  0%|          | 0/17720 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [3]:
# ==============================================================================
# EVALUATION CELL : BLEU-1, BLEU-2, BLEU-3, BLEU-4 and ROUGE-L (NLLB)
# ==============================================================================
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import sacrebleu
from sacrebleu.metrics import BLEU
import evaluate
import os

# --- 1. CONFIGURATIONS ---
DATASET_PATH = "./variants"
dataset_variant = "lower"
MSRC_LANG = "de"
AUX_LANG = "dgs"
RUN = "Vanilla"                      # Update to the correct folder generated during training
SRC_LANG = 'de'
TGT_LANG = 'dgs'
MAX_INPUT_LENGTH = 256
MAX_TGT_LENGTH = 256

# Since inference requires memory primarily for forward calculations, 
# execution on the GPU ensures maximum speed when sweeping the dataset
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
VAR_NAME = dataset_variant

# Paths
VARIANT = '.' + dataset_variant if dataset_variant != 'plain' else '' 
output_dir = f'runs/{MSRC_LANG}-{AUX_LANG}-{VAR_NAME}-run{RUN}'

# ATTENTION: Here the validation file (.val.csv) was kept as per your original script. 
# When running the final results for the paper table, switch to .test.csv
TEST_PATH = os.path.join(DATASET_PATH, f"de-dgs{VARIANT}.test.csv")

print(f" Loading model from: {output_dir}")
print(f" Using test dataset: {TEST_PATH}")

# --- 2. LOAD MODEL AND DATA ---
try:
    # CRITICAL CHANGE 1: The NLLB Tokenizer needs src_lang upon initialization
    tokenizer = AutoTokenizer.from_pretrained(output_dir, src_lang="deu_Latn")
    model = AutoModelForSeq2SeqLM.from_pretrained(output_dir).to(DEVICE)
except OSError:
    print(" ERROR: Model folder not found. Check the 'RUN' variable.")
    raise

# CRITICAL CHANGE 2: Safe extraction of the language ID for NLLB
forced_bos_token_id = tokenizer.convert_tokens_to_ids("deu_Latn")

# Reads the test CSV
df_test = pd.read_csv(TEST_PATH)
df_test = df_test[df_test['de'].notna() & df_test['dgs'].notna()]

# --- 3. GENERATION (INFERENCE) ---
print("\n Generating translations (This may take a few minutes)...")
preds = []
refs = []

# LOCKS MODEL IN DETERMINISTIC MODE (Turns off Dropout and normalizes gradients)
model.eval()

for index, row in tqdm(df_test.iterrows(), total=len(df_test)):
    input_text = str(row[SRC_LANG])
    target_text = str(row[TGT_LANG])
    
    inputs = tokenizer(input_text, return_tensors="pt", max_length=MAX_INPUT_LENGTH, truncation=True).to(DEVICE)
    
    with torch.no_grad():
       outputs = model.generate(
            **inputs, 
            max_length=MAX_TGT_LENGTH,
            forced_bos_token_id=forced_bos_token_id,
            num_beams=1,                  
            early_stopping=True            
        )
        
    pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    preds.append(pred_text)
    refs.append(target_text)

# --- 4. METRICS CALCULATION ---
print("\n Calculating purist metrics...")

refs_transposed = [refs] 
refs_flat = refs

results = {}

# A) SacreBLEU (Canonical) - TOKENIZE NONE MANDATORY
sb = sacrebleu.corpus_bleu(preds, refs_transposed, lowercase=True, tokenize='none')
results['SacreBLEU'] = sb.score

# B) BLEU 1, 2, 3, 4 - TOKENIZE NONE MANDATORY
results['BLEU-1'] = BLEU(lowercase=True, tokenize='none', max_ngram_order=1).corpus_score(preds, refs_transposed).score
results['BLEU-2'] = BLEU(lowercase=True, tokenize='none', max_ngram_order=2).corpus_score(preds, refs_transposed).score
results['BLEU-3'] = BLEU(lowercase=True, tokenize='none', max_ngram_order=3).corpus_score(preds, refs_transposed).score
results['BLEU-4'] = BLEU(lowercase=True, tokenize='none', max_ngram_order=4).corpus_score(preds, refs_transposed).score

# C) ROUGE (Recall-Oriented)
try:
    rouge = evaluate.load("rouge")
    rouge_res = rouge.compute(predictions=preds, references=refs_flat)
    results['ROUGE-L'] = rouge_res['rougeL'] * 100 
except Exception as e:
    print(f" Warning: Could not calculate ROUGE ({e})")
    results['ROUGE-L'] = 0.0

# --- 5. RESULTS DISPLAY ---
print("\n" + "="*40)
print(f" FINAL PERFORMANCE REPORT (NLLB-600M)")
print("="*40)
print(f"{'Metric':<15} | {'Score':<10}")
print("-" * 28)
print(f"{'BLEU-1 ':<15} | {results['BLEU-1']:.2f}")
print(f"{'BLEU-2 ':<15} | {results['BLEU-2']:.2f}")
print(f"{'BLEU-3 ':<15} | {results['BLEU-3']:.2f}")
print(f"{'BLEU-4 ':<15} | {results['BLEU-4']:.2f}")
print("-" * 28)
print(f"{'SacreBLEU':<15} | {results['SacreBLEU']:.2f}")
print(f"{'ROUGE-L':<15} | {results['ROUGE-L']:.2f}")
print("="*40)

# Saves to CSV 
df_result_file = pd.DataFrame({'de': df_test['de'], 'dgs_ref': refs, 'dgs_pred': preds})
save_file = os.path.join(output_dir, 'final_metrics_evaluation.csv')
df_result_file.to_csv(save_file, index=False)
print(f"\nDetailed file with translations saved at: {save_file}")

 Loading model from: runs/de-dgs-lower-runVanilla
 Using test dataset: ./variants/de-dgs.lower.test.csv

 Generating translations (This may take a few minutes)...


  0%|          | 0/642 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



 Calculating purist metrics...

 FINAL PERFORMANCE REPORT (NLLB-600M)
Metric          | Score     
----------------------------
BLEU-1          | 48.10
BLEU-2          | 29.50
BLEU-3          | 19.23
BLEU-4          | 12.95
----------------------------
SacreBLEU       | 12.95
ROUGE-L         | 45.10

Detailed file with translations saved at: runs/de-dgs-lower-runVanilla/final_metrics_evaluation.csv
